# 05 · Merge survey and household records safely

**Module:** 2 · **Time:** 45 minutes · **Prerequisites:** Selecting, filtering and creating variables

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/05_merging.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F05_merging.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** The repository is private and badges point to `main` after merge.

## Learning goals

- **J**oin a person-level survey with a household lookup.
- **V**alidate many-to-one keys and diagnose unmatched records.
- **D**istinguish joining columns from appending rows.

In [ ]:
import numpy as np
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print("Fictional observations:", len(survey))

## Why this matters

Researchers often combine individual wages with household characteristics. A merge without key checks can duplicate observations and distort regression samples. The fictional survey deliberately contains duplicate `person_id` values; the household lookup is uniquely keyed on `household_id`.

## Make the household lookup

The household file has one row per household, while the respondent survey can have multiple people in each household.

In [ ]:
households = pd.DataFrame({
    "household_id": ["H001", "H002", "H003"],
    "housing_type": ["rent", "own", "rent"],
    "urban": [True, False, True],
})
assert households["household_id"].is_unique
print(households)

## Merge with validation and an indicator

`validate='many_to_one'` enforces that the lookup is unique. `indicator=True` labels matched and unmatched rows.

In [ ]:
merged = survey.merge(
    households, on="household_id", how="left",
    validate="many_to_one", indicator=True
)
print(merged["_merge"].value_counts())
print("Rows preserved:", len(merged) == len(survey))

## Diagnose invalid keys

We deliberately duplicate a lookup key, then catch the specific error so the lesson remains executable. Never silence a real merge problem by dropping duplicates at random.

In [ ]:
bad_lookup = pd.concat([households, households.iloc[[0]]], ignore_index=True)
try:
    survey.merge(bad_lookup, on="household_id", validate="many_to_one")
except pd.errors.MergeError:
    print("Expected problem: household lookup key is not unique.")

## Append rows, do not confuse with merge

`concat` stacks compatible datasets by row. Use it for new survey waves only when variable definitions and units agree.

In [ ]:
one_more = survey.iloc[[0]].copy()
appended = pd.concat([survey, one_more], ignore_index=True)
print("Original / appended counts:", len(survey), len(appended))
print("Duplicated respondent IDs:", appended["person_id"].duplicated().sum())

## If you know Stata/R

| Task | pandas | Stata | R |
| --- | --- | --- | --- |
| Many-to-one merge | `merge(..., validate='many_to_one')` | `merge m:1 household_id using ...` | `left_join(..., relationship='many-to-one')` |
| Merge status | `indicator=True` | `_merge` | `anti_join` for unmatched |
| Append rows | `pd.concat([a, b])` | `append using ...` | `dplyr::bind_rows(a, b)` |

Check both key uniqueness and row counts. The correct relationship depends on the dataset's unit of observation.

## Try it

Complete the answer cells, then run the friendly checks. Blank learner exercises produce hints rather than exceptions.

### ★ Exercise 1 — Validate the household key

Set `unique_households` to a Boolean checking whether `households['household_id']` has unique values.

In [ ]:
unique_households = None

In [ ]:
if isinstance(unique_households, (bool, np.bool_)) and unique_households:
    print("✓ Household key is unique")
else:
    print("Hint: use the Series.is_unique property.")

### ★★ Exercise 2 — Merge person and household data

Create `survey_with_household` using a left many-to-one merge, preserving all survey rows.

In [ ]:
survey_with_household = None

In [ ]:
expected = survey.merge(households, on="household_id", how="left", validate="many_to_one")
if isinstance(survey_with_household, pd.DataFrame) and survey_with_household.equals(expected):
    print("✓ Household merge correct")
else:
    print("Hint: merge on household_id and validate many_to_one.")

### ★★ Exercise 3 — Count unmatched rows

Set `unmatched_count` to the count of rows whose `_merge` indicator is `left_only` in `merged`.

In [ ]:
unmatched_count = None

In [ ]:
expected = int(merged["_merge"].eq("left_only").sum())
if isinstance(unmatched_count, (int, np.integer)) and unmatched_count == expected:
    print("✓ Unmatched count correct")
else:
    print("Hint: compare merged['_merge'] with 'left_only' and sum.")

### ★★★ Exercise 4 — Append a new respondent row

Create `extended_survey` by appending `survey.iloc[[0]]` to `survey` with a fresh RangeIndex. Do not merge it as a column.

In [ ]:
extended_survey = None

In [ ]:
expected = pd.concat([survey, survey.iloc[[0]]], ignore_index=True)
if isinstance(extended_survey, pd.DataFrame) and extended_survey.equals(expected):
    print("✓ Append operation correct")
else:
    print("Hint: pd.concat stacks the two DataFrames; use ignore_index=True.")

## Common errors

- `MergeError` from `validate` means the key relation is wrong; inspect both sides rather than suppressing the check.
- Unexpected extra rows usually indicate a many-to-many merge; count keys before and after.
- `KeyError` means a join column is missing or differently named.
- An unmatched status does not necessarily mean a data error; investigate the population and coverage.

## Recap

- Specify the unit of observation and check merge keys.
- Use `validate` and `indicator` to audit joins.
- A merge attaches columns; `concat` stacks rows.
- Investigate duplicates and unmatched cases rather than silently deleting them.

## Next steps

Next: [Reshape wide and long surveys](06_reshaping.ipynb). Reference: [pandas merge guide](https://pandas.pydata.org/docs/user_guide/merging.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md).